# 3.2 Probability Rules

**Goal:** translate “or,” “and,” disjointness, and exhaustive coverage into the correct probability operations.

**Prerequisites:** [Probability Fundamentals](1_Probability-Fundamentals.ipynb) and set notation. **Runtime:** Python 3.10+, standard library only. Idealized examples and saved outputs are included; run cells in order.

## Contents
1. Union of events
2. Intersection of events
3. Addition rule
4. Mutually exclusive events
5. Exhaustive events
6. Multiplication rule
7. ML alert example and sanity checks
8. Practice and solutions

## 1. Union of Events

The union $A\cup B$ occurs when A happens, B happens, **or both** happen. Probability uses this inclusive meaning of “or” unless the question explicitly says “exactly one.”

**Worked example:** on a fair die, $A=\{2,4,6\}$ (even) and $B=\{4,5,6\}$ (at least 4). Their union is {2,4,5,6}, giving probability 4/6.

The event “exactly one of A or B” is the symmetric difference $(A\setminus B)\cup(B\setminus A)$. It excludes the intersection and is a different event.

## 2. Intersection of Events

The intersection $A\cap B$ means **both** A and B occur in the same experiment. In the die example, $A\cap B=\{4,6\}$, probability 2/6.

The word “both” does not imply independent events. Independence is a separate relationship between probabilities. When a question says “A then B,” define the complete experiment and the time-specific events before multiplying.

In [1]:
from fractions import Fraction
omega = set(range(1, 7))
A, B = {2, 4, 6}, {4, 5, 6}
def probability(event):
    if not event <= omega:
        raise ValueError("Event must be a subset of the die sample space")
    return Fraction(len(event), len(omega))

print("Union:", sorted(A | B), probability(A | B))
print("Intersection:", sorted(A & B), probability(A & B))
print("Exactly one:", sorted(A ^ B), probability(A ^ B))

Union: [2, 4, 5, 6] 2/3
Intersection: [4, 6] 1/3
Exactly one: [2, 5] 1/3


## 3. Addition Rule

For any two events,
$$P(A\cup B)=P(A)+P(B)-P(A\cap B).$$
The intersection is counted twice in the two individual probabilities, so subtract it once. The rule does not assume independence.

**Worked example:** $P(A)=3/6$, $P(B)=3/6$, and $P(A\cap B)=2/6$, so the union is $3/6+3/6-2/6=4/6$.

For three events, inclusion-exclusion gives
$$P(A\cup B\cup C)=P(A)+P(B)+P(C)-P(A\cap B)-P(A\cap C)-P(B\cap C)+P(A\cap B\cap C).$$

For “exactly one of two”:
$$P(A\mathbin{\triangle}B)=P(A)+P(B)-2P(A\cap B).$$

Useful checks are $\max(P(A),P(B))\le P(A\cup B)\le\min(1,P(A)+P(B))$ and
$$\max(0,P(A)+P(B)-1)\le P(A\cap B)\le\min(P(A),P(B)).$$
An intersection or union outside those bounds indicates inconsistent inputs or arithmetic.

In [2]:
union_via_rule = probability(A)+probability(B)-probability(A & B)
print("Union via addition rule:", union_via_rule)
assert union_via_rule == probability(A | B)
assert probability(A ^ B) == probability(A)+probability(B)-2*probability(A & B)
C = {1, 2, 3}
three_union = (probability(A)+probability(B)+probability(C)
               -probability(A & B)-probability(A & C)-probability(B & C)
               +probability(A & B & C))
print("Three-event union:", three_union)
assert three_union == probability(A | B | C)

Union via addition rule: 2/3
Three-event union: 1


## 4. Mutually Exclusive Events

Events are mutually exclusive (disjoint) when $A\cap B=\varnothing$. They cannot occur together. Their addition rule simplifies to $P(A\cup B)=P(A)+P(B)$.

**Worked example:** a single die cannot be both 1 and 6, so “roll 1” and “roll 6” are disjoint. “Even” and “at least 4” are not disjoint because they share 4 and 6.

Disjointness is not independence. If disjoint events each have positive probability, occurrence of one makes the other impossible, so they are dependent. For a disjoint collection, pairwise disjoint means every pair has empty intersection; this supports summing probabilities over the collection.

## 5. Exhaustive Events

A collection $A_1,\ldots,A_k$ is exhaustive if $\bigcup_iA_i=\Omega$: at least one event always occurs. Exhaustive events need not be disjoint.

A **partition** is both exhaustive and pairwise disjoint, with nonempty cells. Exactly one partition cell contains each outcome. The probabilities of a finite partition sum to 1. An exhaustive overlapping collection can have probabilities summing to more than 1.

**Worked example:** on a die, even and odd form a partition. Events “at most 4” and “at least 3” are exhaustive but overlap at 3 and 4. Their probabilities sum to $4/6+4/6=4/3$, even though their union has probability 1. A sum exceeding 1 is not an error when it adds overlapping event probabilities rather than claiming a single-event probability.

Disjointness alone does not guarantee exhaustiveness: {1} and {2} leave outcomes 3–6 uncovered. Exhaustiveness alone does not justify simple probability addition.

In [3]:
even, odd = {2, 4, 6}, {1, 3, 5}
low, high = {1, 2, 3, 4}, {3, 4, 5, 6}
print("Even/odd disjoint:", not (even & odd), "exhaustive:", even | odd == omega)
print("Low/high disjoint:", not (low & high), "exhaustive:", low | high == omega)
print("Overlapping exhaustive probability sum:", probability(low)+probability(high))
assert probability(even)+probability(odd) == 1
assert probability(low | high) == 1

Even/odd disjoint: True exhaustive: True
Low/high disjoint: False exhaustive: True
Overlapping exhaustive probability sum: 4/3


## 6. Multiplication Rule

For $P(A)>0$, define the conditional probability $P(B\mid A)=P(A\cap B)/P(A)$. Rearranging gives
$$P(A\cap B)=P(A)P(B\mid A).$$
Similarly, if $P(B)>0$, $P(A\cap B)=P(B)P(A\mid B)$. Module 3.3 develops conditioning in detail.

If A and B are independent, $P(B\mid A)=P(B)$ where defined, giving $P(A\cap B)=P(A)P(B)$. Multiplying marginal probabilities without this assumption is generally wrong.

**Worked example: without replacement.** Draw two balls from a bag with 3 red and 2 blue. The chance of two reds is $(3/5)(2/4)=3/10$: after a red first draw, the bag's composition changes.

**With replacement and a fresh uniform draw,** the two-red probability is $(3/5)^2=9/25$. Replacement and the fresh-draw mechanism justify using the same conditional probability again. The two procedures answer different questions.

For repeated outcomes, the chain rule uses each conditional history. “And” corresponds to an intersection, but does not automatically imply a product of unconditional probabilities.

In [4]:
from itertools import permutations
balls = ["R1", "R2", "R3", "B1", "B2"]
ordered_draws = list(permutations(balls, 2))
two_red = [pair for pair in ordered_draws if all(ball.startswith("R") for ball in pair)]
without_replacement = Fraction(3, 5)*Fraction(2, 4)
with_replacement = Fraction(3, 5)**2
print("Two reds without replacement:", without_replacement)
print("Two reds with replacement:", with_replacement)
assert without_replacement == Fraction(len(two_red), len(ordered_draws))

Two reds without replacement: 3/10
Two reds with replacement: 9/25


## 7. AI/ML application: combining alerts

Suppose 30% of transactions trigger rule A, 20% trigger rule B, and 10% trigger both. A system that reviews “A or B” reviews 40%; “A and B” reviews 10%; exactly one rule reviews 30%. Adding the two flag rates as 50% double counts the overlap.

Independence would predict an intersection $0.30\times0.20=0.06$, but the stated intersection is 0.10. Correlated features and alert rules are common; do not assume independence just because systems have different names.

These are trigger probabilities, not fraud probabilities. To estimate the chance of fraud given an alert, labeled outcomes and conditional/Bayesian reasoning are needed. Also confirm that rule rates use the same population and time window.

In [5]:
p_a, p_b, p_both = Fraction(3, 10), Fraction(1, 5), Fraction(1, 10)
p_either = p_a+p_b-p_both
p_exactly_one = p_a+p_b-2*p_both
print("Review either rule:", float(p_either))
print("Review both rules:", float(p_both))
print("Review exactly one:", float(p_exactly_one))
print("Independent-intersection prediction:", float(p_a*p_b))
assert p_either == Fraction(2, 5)
assert p_both != p_a*p_b

Review either rule: 0.4
Review both rules: 0.1
Review exactly one: 0.3
Independent-intersection prediction: 0.06


## Practice

1. Given $P(A)=0.6$, $P(B)=0.5$, $P(A\cap B)=0.3$, find union, neither, and exactly one.
2. On one die, are “even” and “odd” disjoint, exhaustive, independent?
3. Give an exhaustive but overlapping pair of events on a die.
4. Draw two cards uniformly without replacement from a standard 52-card deck. Find the probability both are aces.
5. Is an intersection of 0.7 consistent with marginals 0.4 and 0.8?
6. If two independent sensors each miss a fault with probability 0.1, what is the chance at least one detects it? What assumption must be checked?
7. Why is a 30% alert rate not automatically a 30% fraud probability given the alert?

## Solutions

1. Union $0.6+0.5-0.3=0.8$; neither $1-0.8=0.2$; exactly one $0.6+0.5-2(0.3)=0.5$.
2. Disjoint and exhaustive, but not independent: intersection 0 differs from $(1/2)(1/2)=1/4$.
3. At most 4 and at least 3 cover every die face but overlap at 3 and 4.
4. $(4/52)(3/51)=1/221$. The second factor conditions on an ace being removed.
5. No. An intersection cannot exceed the smaller marginal, 0.4.
6. $1-0.1^2=0.99$ under independent misses conditional on a fault. Shared failure causes can invalidate that conditional independence.
7. It is the chance of an alert among transactions, whereas the requested quantity conditions on an alert and asks about fraud. Those reverse the roles of event and condition.

**Readiness check:** distinguish inclusive “or” from exactly one, disjoint from independent, and exhaustive collections from partitions.

**Next:** [3.3 Conditional Probability](3_Conditional-Probability.ipynb).